# Classifying Pet Images: run the real models

This notebook runs the project end to end with the pretrained CNNs:
ResNet-18, AlexNet and VGG-16 (the three the Udacity project compares), plus
ResNet-50 and EfficientNet-B0. At the end you download the output files and
commit them to the repo.

**Tip:** for a faster run, choose *Runtime → Change runtime type → GPU* first.
It also works on the free CPU runtime; it just takes a few minutes longer.

## 1. Get the code

In [ ]:
!git clone --depth 1 https://github.com/thomasmd321/AI_Programming_with_Python_Project_1.git
%cd AI_Programming_with_Python_Project_1/workspace

## 2. Install requirements and check for a GPU
Colab already has PyTorch, so this is quick.

In [ ]:
!pip install -q -r ../requirements.txt
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (using CPU)")

## 3. Smoke test
Classifies one collie image with VGG-16 (downloads the VGG weights the first time).

In [ ]:
!python test_classifier.py

## 4. Run every model on both image folders
Each run writes `<model>_pet-images.txt` / `.csv` (or `_uploaded-images`),
then the script prints the comparison table.

In [ ]:
!sh run_models_batch.sh

In [ ]:
!sh run_models_batch_uploaded.sh

## 5. Look at the results
One row per image per model, from the CSV files.

In [ ]:
import glob
import pandas as pd

frames = []
for path in sorted(glob.glob("*_*-images.csv")):
    df = pd.read_csv(path)
    df["images"] = "uploaded" if "uploaded" in path else "pet"
    frames.append(df)
results = pd.concat(frames, ignore_index=True)

# Share of images where the classifier label matches the filename label.
summary = (results.groupby(["images", "model"])
           .agg(images_n=("filename", "size"),
                pct_match=("labels_match", "mean"),
                avg_confidence=("confidence", "mean"))
           .assign(pct_match=lambda d: (d.pct_match * 100).round(1),
                   avg_confidence=lambda d: (d.avg_confidence * 100).round(1)))
summary

In [ ]:
# Images the models got wrong, with their top guesses.
wrong = results[results.labels_match == 0]
wrong[["images", "model", "filename", "pet_label", "top_guesses"]]

## 6. Download the outputs
Downloads `results.zip`. To add the results to the repo, unzip it into
`workspace/` in your local clone (replacing the old files), then commit and
push, for example:

```
git add workspace/*-images.txt workspace/*-images.csv
git commit -m "Update model outputs from a real run"
git push
```

In [ ]:
!zip -q ../results.zip *_pet-images.txt *_pet-images.csv *_uploaded-images.txt *_uploaded-images.csv
from google.colab import files
files.download("../results.zip")